In [0]:
import requests
import json
import os
from datetime import date

BASE_URL = "https://archive-api.open-meteo.com/v1/archive"

print("European Weather & Climate Change Analytics Lakehouse")
print("Data Source: Open-Meteo Historical Weather API")
print("Model: ERA5")
print("Production Period: 1940-01-01 to 2025-12-31")
print("Production Cities: 20")

European Weather & Climate Change Analytics Lakehouse
Data Source: Open-Meteo Historical Weather API
Model: ERA5
Production Period: 1940-01-01 to 2025-12-31
Production Cities: 20


In [0]:
cities = [
    {
        "city": "London",
        "country": "United Kingdom",
        "region": "Western Europe",
        "latitude": 51.5074,
        "longitude": -0.1278
    },
    {
        "city": "Oslo",
        "country": "Norway",
        "region": "Northern Europe",
        "latitude": 59.9139,
        "longitude": 10.7522
    },
    {
        "city": "Berlin",
        "country": "Germany",
        "region": "Central Europe",
        "latitude": 52.5200,
        "longitude": 13.4050
    },
    {
        "city": "Madrid",
        "country": "Spain",
        "region": "Southern Europe",
        "latitude": 40.4168,
        "longitude": -3.7038
    },
    {
        "city": "Warsaw",
        "country": "Poland",
        "region": "Eastern Europe",
        "latitude": 52.2297,
        "longitude": 21.0122
    }
]

print("Number of sample cities:", len(cities))

for city in cities:
    print(city["city"], "-", city["country"], "-", city["region"])

Number of sample cities: 5
London - United Kingdom - Western Europe
Oslo - Norway - Northern Europe
Berlin - Germany - Central Europe
Madrid - Spain - Southern Europe
Warsaw - Poland - Eastern Europe


In [0]:
weather_variables = [
    "temperature_2m",
    "relative_humidity_2m",
    "apparent_temperature",
    "precipitation",
    "rain",
    "snowfall",
    "weather_code",
    "pressure_msl",
    "surface_pressure",
    "cloud_cover",
    "wind_speed_10m",
    "wind_direction_10m"
]

print("Number of weather variables:", len(weather_variables))

for variable in weather_variables:
    print(variable)

Number of weather variables: 12
temperature_2m
relative_humidity_2m
apparent_temperature
precipitation
rain
snowfall
weather_code
pressure_msl
surface_pressure
cloud_cover
wind_speed_10m
wind_direction_10m


In [0]:
def get_weather_data(city_info, start_date, end_date):
    
    params = {
        "latitude": city_info["latitude"],
        "longitude": city_info["longitude"],
        "start_date": start_date,
        "end_date": end_date,
        "hourly": ",".join(weather_variables),
        "timezone": "UTC",
        "models": "era5"
    }
    
    response = requests.get(
        BASE_URL,
        params=params,
        timeout=120
    )
    
    response.raise_for_status()
    
    return response.json()

In [0]:
full_load_sample = []

for city in cities:
    
    print("Downloading:", city["city"])
    
    data = get_weather_data(
        city,
        "2024-01-01",
        "2024-12-31"
    )
    
    hourly = data["hourly"]
    
    number_of_hours = len(hourly["time"])
    
    print("Hours received:", number_of_hours)
    
    for i in range(number_of_hours):
        
        record = {
            "city": city["city"],
            "country": city["country"],
            "region": city["region"],
            "timestamp": hourly["time"][i],
            "temperature_2m": hourly["temperature_2m"][i],
            "relative_humidity_2m": hourly["relative_humidity_2m"][i],
            "apparent_temperature": hourly["apparent_temperature"][i],
            "precipitation": hourly["precipitation"][i],
            "rain": hourly["rain"][i],
            "snowfall": hourly["snowfall"][i],
            "weather_code": hourly["weather_code"][i],
            "pressure_msl": hourly["pressure_msl"][i],
            "surface_pressure": hourly["surface_pressure"][i],
            "cloud_cover": hourly["cloud_cover"][i],
            "wind_speed_10m": hourly["wind_speed_10m"][i],
            "wind_direction_10m": hourly["wind_direction_10m"][i]
        }
        
        full_load_sample.append(record)

print("\nTOTAL FULL LOAD SAMPLE ROWS:", len(full_load_sample))

Downloading: London
Hours received: 8784
Downloading: Oslo
Hours received: 8784
Downloading: Berlin
Hours received: 8784
Downloading: Madrid
Hours received: 8784
Downloading: Warsaw
Hours received: 8784

TOTAL FULL LOAD SAMPLE ROWS: 43920


In [0]:
from collections import Counter

full_counts = Counter(
    record["city"] for record in full_load_sample
)

print("FULL LOAD VALIDATION")
print("--------------------")

for city, count in full_counts.items():
    print(city, ":", count)

print("\nTotal rows:", len(full_load_sample))
print("Number of columns:", len(full_load_sample[0]))
print("First timestamp:", min(x["timestamp"] for x in full_load_sample))
print("Last timestamp:", max(x["timestamp"] for x in full_load_sample))

FULL LOAD VALIDATION
--------------------
London : 8784
Oslo : 8784
Berlin : 8784
Madrid : 8784
Warsaw : 8784

Total rows: 43920
Number of columns: 16
First timestamp: 2024-01-01T00:00
Last timestamp: 2024-12-31T23:00


In [0]:
full_file = "/tmp/full_load_sample_5_cities_2024.json"

with open(full_file, "w") as f:
    json.dump(full_load_sample, f)

full_bytes = os.path.getsize(full_file)
full_kb = full_bytes / 1024
full_mb = full_bytes / (1024 ** 2)

print("FULL LOAD FILE SAVED")
print("Path:", full_file)
print("Rows:", len(full_load_sample))
print("Size KB:", round(full_kb, 2))
print("Size MB:", round(full_mb, 2))

FULL LOAD FILE SAVED
Path: /tmp/full_load_sample_5_cities_2024.json
Rows: 43920
Size KB: 16171.03
Size MB: 15.79


In [0]:
incremental_sample = []

for city in cities:
    
    print("Downloading:", city["city"])
    
    data = get_weather_data(
        city,
        "2025-01-01",
        "2025-01-07"
    )
    
    hourly = data["hourly"]
    
    number_of_hours = len(hourly["time"])
    
    print("Hours received:", number_of_hours)
    
    for i in range(number_of_hours):
        
        record = {
            "city": city["city"],
            "country": city["country"],
            "region": city["region"],
            "timestamp": hourly["time"][i],
            "temperature_2m": hourly["temperature_2m"][i],
            "relative_humidity_2m": hourly["relative_humidity_2m"][i],
            "apparent_temperature": hourly["apparent_temperature"][i],
            "precipitation": hourly["precipitation"][i],
            "rain": hourly["rain"][i],
            "snowfall": hourly["snowfall"][i],
            "weather_code": hourly["weather_code"][i],
            "pressure_msl": hourly["pressure_msl"][i],
            "surface_pressure": hourly["surface_pressure"][i],
            "cloud_cover": hourly["cloud_cover"][i],
            "wind_speed_10m": hourly["wind_speed_10m"][i],
            "wind_direction_10m": hourly["wind_direction_10m"][i]
        }
        
        incremental_sample.append(record)

print("\nTOTAL INCREMENTAL SAMPLE ROWS:", len(incremental_sample))

Downloading: London
Hours received: 168
Downloading: Oslo
Hours received: 168
Downloading: Berlin
Hours received: 168
Downloading: Madrid
Hours received: 168
Downloading: Warsaw
Hours received: 168

TOTAL INCREMENTAL SAMPLE ROWS: 840


In [0]:
incremental_counts = Counter(
    record["city"] for record in incremental_sample
)

print("INCREMENTAL LOAD VALIDATION")
print("---------------------------")

for city, count in incremental_counts.items():
    print(city, ":", count)

print("\nTotal rows:", len(incremental_sample))
print("Number of columns:", len(incremental_sample[0]))
print("First timestamp:", min(x["timestamp"] for x in incremental_sample))
print("Last timestamp:", max(x["timestamp"] for x in incremental_sample))

INCREMENTAL LOAD VALIDATION
---------------------------
London : 168
Oslo : 168
Berlin : 168
Madrid : 168
Warsaw : 168

Total rows: 840
Number of columns: 16
First timestamp: 2025-01-01T00:00
Last timestamp: 2025-01-07T23:00


In [0]:
incremental_file = "/tmp/incremental_load_sample_5_cities_2025_01_01_to_07.json"

with open(incremental_file, "w") as f:
    json.dump(incremental_sample, f)

incremental_bytes = os.path.getsize(incremental_file)
incremental_kb = incremental_bytes / 1024
incremental_mb = incremental_bytes / (1024 ** 2)

print("INCREMENTAL FILE SAVED")
print("Path:", incremental_file)
print("Rows:", len(incremental_sample))
print("Size KB:", round(incremental_kb, 2))
print("Size MB:", round(incremental_mb, 3))

INCREMENTAL FILE SAVED
Path: /tmp/incremental_load_sample_5_cities_2025_01_01_to_07.json
Rows: 840
Size KB: 309.44
Size MB: 0.302


In [0]:
full_df = spark.createDataFrame(full_load_sample)

incremental_df = spark.createDataFrame(incremental_sample)

print("Full Load Spark rows:", full_df.count())
print("Incremental Spark rows:", incremental_df.count())
print("Number of columns:", len(full_df.columns))

Full Load Spark rows: 43920
Incremental Spark rows: 840
Number of columns: 16


In [0]:
full_df.printSchema()

root
 |-- apparent_temperature: double (nullable = true)
 |-- city: string (nullable = true)
 |-- cloud_cover: long (nullable = true)
 |-- country: string (nullable = true)
 |-- precipitation: double (nullable = true)
 |-- pressure_msl: double (nullable = true)
 |-- rain: double (nullable = true)
 |-- region: string (nullable = true)
 |-- relative_humidity_2m: long (nullable = true)
 |-- snowfall: double (nullable = true)
 |-- surface_pressure: double (nullable = true)
 |-- temperature_2m: double (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- weather_code: long (nullable = true)
 |-- wind_direction_10m: long (nullable = true)
 |-- wind_speed_10m: double (nullable = true)



In [0]:
display(
    full_df
    .orderBy("city", "timestamp")
    .limit(20)
)

apparent_temperature,city,cloud_cover,country,precipitation,pressure_msl,rain,region,relative_humidity_2m,snowfall,surface_pressure,temperature_2m,timestamp,weather_code,wind_direction_10m,wind_speed_10m
1.9,Berlin,98,Germany,0.0,1004.8,0.0,Central Europe,88,0.0,1000.3,5.4,2024-01-01T00:00,3,206,14.0
1.9,Berlin,97,Germany,0.0,1004.7,0.0,Central Europe,87,0.0,1000.2,5.3,2024-01-01T01:00,3,203,12.9
1.7,Berlin,98,Germany,0.0,1004.6,0.0,Central Europe,87,0.0,1000.0,5.1,2024-01-01T02:00,3,202,12.4
1.2,Berlin,96,Germany,0.0,1004.8,0.0,Central Europe,87,0.0,1000.2,4.8,2024-01-01T03:00,3,202,13.6
1.1,Berlin,91,Germany,0.0,1004.6,0.0,Central Europe,86,0.0,1000.0,4.7,2024-01-01T04:00,3,206,13.2
1.0,Berlin,90,Germany,0.1,1004.5,0.1,Central Europe,86,0.0,999.9,4.6,2024-01-01T05:00,51,207,12.9
1.2,Berlin,78,Germany,0.1,1004.3,0.1,Central Europe,87,0.0,999.7,4.7,2024-01-01T06:00,51,204,12.6
1.3,Berlin,77,Germany,0.1,1004.7,0.1,Central Europe,86,0.0,1000.1,4.9,2024-01-01T07:00,51,200,13.4
1.6,Berlin,71,Germany,0.1,1005.0,0.1,Central Europe,86,0.0,1000.4,5.3,2024-01-01T08:00,51,203,14.5
2.3,Berlin,66,Germany,0.0,1005.2,0.0,Central Europe,84,0.0,1000.7,6.0,2024-01-01T09:00,2,210,15.3


In [0]:
from pyspark.sql.functions import col, sum as spark_sum

null_counts = full_df.select(
    [
        spark_sum(col(c).isNull().cast("int")).alias(c)
        for c in full_df.columns
    ]
)

display(null_counts)

apparent_temperature,city,cloud_cover,country,precipitation,pressure_msl,rain,region,relative_humidity_2m,snowfall,surface_pressure,temperature_2m,timestamp,weather_code,wind_direction_10m,wind_speed_10m
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [0]:
duplicate_count = (
    full_df
    .groupBy("city", "timestamp")
    .count()
    .filter(col("count") > 1)
    .count()
)

print("Duplicate city/timestamp combinations:", duplicate_count)

Duplicate city/timestamp combinations: 0


In [0]:
production_start = date(1940, 1, 1)
production_end = date(2025, 12, 31)

number_of_days = (
    production_end - production_start
).days + 1

production_cities = 20
hours_per_day = 24

estimated_full_rows = (
    number_of_days
    * hours_per_day
    * production_cities
)

print("Production period days:", number_of_days)
print("Production cities:", production_cities)
print("Estimated Full Load rows:", f"{estimated_full_rows:,}")

Production period days: 31412
Production cities: 20
Estimated Full Load rows: 15,077,760


In [0]:
daily_incremental_rows = production_cities * 24

weekly_incremental_rows = daily_incremental_rows * 7

monthly_incremental_rows = daily_incremental_rows * 30

yearly_incremental_rows = daily_incremental_rows * 365.25

print("Daily incremental rows:", f"{daily_incremental_rows:,}")
print("Weekly incremental rows:", f"{weekly_incremental_rows:,}")
print("Monthly incremental rows:", f"{monthly_incremental_rows:,}")
print("Yearly incremental rows:", f"{int(yearly_incremental_rows):,}")

Daily incremental rows: 480
Weekly incremental rows: 3,360
Monthly incremental rows: 14,400
Yearly incremental rows: 175,320


In [0]:
bytes_per_record = full_bytes / len(full_load_sample)

estimated_production_bytes = (
    bytes_per_record * estimated_full_rows
)

estimated_production_mb = (
    estimated_production_bytes / (1024 ** 2)
)

estimated_production_gb = (
    estimated_production_bytes / (1024 ** 3)
)

print("Measured sample size:", round(full_mb, 2), "MB")
print("Average bytes per record:", round(bytes_per_record, 2))

print(
    "Estimated production raw size:",
    round(estimated_production_mb, 2),
    "MB"
)

print(
    "Estimated production raw size:",
    round(estimated_production_gb, 2),
    "GB"
)

Measured sample size: 15.79 MB
Average bytes per record: 377.03
Estimated production raw size: 5421.41 MB
Estimated production raw size: 5.29 GB


In [0]:
estimated_daily_bytes = (
    bytes_per_record * daily_incremental_rows
)

estimated_weekly_bytes = (
    bytes_per_record * weekly_incremental_rows
)

estimated_monthly_bytes = (
    bytes_per_record * monthly_incremental_rows
)

estimated_yearly_bytes = (
    bytes_per_record * yearly_incremental_rows
)

print(
    "Daily Increment:",
    round(estimated_daily_bytes / 1024, 2),
    "KB"
)

print(
    "Weekly Increment:",
    round(estimated_weekly_bytes / (1024 ** 2), 3),
    "MB"
)

print(
    "Monthly Increment:",
    round(estimated_monthly_bytes / (1024 ** 2), 3),
    "MB"
)

print(
    "Yearly Increment:",
    round(estimated_yearly_bytes / (1024 ** 2), 2),
    "MB"
)

Daily Increment: 176.73 KB
Weekly Increment: 1.208 MB
Monthly Increment: 5.178 MB
Yearly Increment: 63.04 MB


In [0]:
volume_summary = [
    (
        "Phase 1 Full Sample",
        "5 cities, full year 2024",
        len(full_load_sample),
        round(full_mb, 3)
    ),
    (
        "Phase 1 Incremental Sample",
        "5 cities, Jan 1-7 2025",
        len(incremental_sample),
        round(incremental_mb, 3)
    ),
    (
        "Production Full Load",
        "20 cities, 1940-2025",
        estimated_full_rows,
        round(estimated_production_mb, 2)
    ),
    (
        "Production Daily Increment",
        "20 cities, 1 new day",
        daily_incremental_rows,
        round(
            estimated_daily_bytes / (1024 ** 2),
            4
        )
    )
]

summary_df = spark.createDataFrame(
    volume_summary,
    [
        "load_type",
        "description",
        "rows",
        "estimated_size_mb"
    ]
)

display(summary_df)

load_type,description,rows,estimated_size_mb
Phase 1 Full Sample,"5 cities, full year 2024",43920,15.792
Phase 1 Incremental Sample,"5 cities, Jan 1-7 2025",840,0.302
Production Full Load,"20 cities, 1940-2025",15077760,5421.41
Production Daily Increment,"20 cities, 1 new day",480,0.1726


In [0]:
print("=" * 60)
print("PHASE 1 DATA VERIFICATION")
print("=" * 60)

print("\nDATA SOURCE")
print("Open-Meteo Historical Weather API")
print("Model: ERA5")

print("\nFULL LOAD SAMPLE")
print("Cities: 5")
print("Period: 2024-01-01 to 2024-12-31")
print("Rows:", f"{len(full_load_sample):,}")
print("Size:", round(full_mb, 2), "MB")

print("\nINCREMENTAL SAMPLE")
print("Cities: 5")
print("Period: 2025-01-01 to 2025-01-07")
print("Rows:", f"{len(incremental_sample):,}")
print("Size:", round(incremental_mb, 3), "MB")

print("\nPRODUCTION DESIGN")
print("Cities: 20")
print("Period: 1940-2025")
print("Estimated Full Load Rows:", f"{estimated_full_rows:,}")
print("Daily Incremental Rows:", f"{daily_incremental_rows:,}")

print("\nVALIDATION")
print("Full Sample Columns:", len(full_df.columns))
print("Duplicate city/timestamps:", duplicate_count)

print("\nFULL → INCREMENTAL CONTINUITY")
print(
    "Full Load ends:",
    max(x["timestamp"] for x in full_load_sample)
)
print(
    "Incremental starts:",
    min(x["timestamp"] for x in incremental_sample)
)

print("\nSTATUS: PHASE 1 DATA PREPARATION COMPLETE")

PHASE 1 DATA VERIFICATION

DATA SOURCE
Open-Meteo Historical Weather API
Model: ERA5

FULL LOAD SAMPLE
Cities: 5
Period: 2024-01-01 to 2024-12-31
Rows: 43,920
Size: 15.79 MB

INCREMENTAL SAMPLE
Cities: 5
Period: 2025-01-01 to 2025-01-07
Rows: 840
Size: 0.302 MB

PRODUCTION DESIGN
Cities: 20
Period: 1940-2025
Estimated Full Load Rows: 15,077,760
Daily Incremental Rows: 480

VALIDATION
Full Sample Columns: 16
Duplicate city/timestamps: 0

FULL → INCREMENTAL CONTINUITY
Full Load ends: 2024-12-31T23:00
Incremental starts: 2025-01-01T00:00

STATUS: PHASE 1 DATA PREPARATION COMPLETE


In [0]:
import json
import os

os.makedirs("/Workspace/Users", exist_ok=True)

print("Data is still available:")
print("Full:", len(full_load_sample))
print("Incremental:", len(incremental_sample))

Data is still available:
Full: 43920
Incremental: 840


In [0]:
import gzip
import json
import os

full_gz = "/tmp/full_load_sample_5_cities_2024.json.gz"
inc_gz = "/tmp/incremental_load_sample_5_cities_2025_01_01_to_07.json.gz"

with gzip.open(full_gz, "wt", encoding="utf-8") as f:
    json.dump(full_load_sample, f)

with gzip.open(inc_gz, "wt", encoding="utf-8") as f:
    json.dump(incremental_sample, f)

print("Full compressed size:",
      round(os.path.getsize(full_gz) / (1024**2), 2), "MB")

print("Incremental compressed size:",
      round(os.path.getsize(inc_gz) / 1024, 2), "KB")

Full compressed size: 0.9 MB
Incremental compressed size: 19.17 KB


In [0]:
import base64
from IPython.display import HTML, display

def download_file(file_path, download_name):
    with open(file_path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode()

    display(HTML(
        f'<a href="data:application/gzip;base64,{encoded}" '
        f'download="{download_name}">'
        f'⬇ Download {download_name}</a>'
    ))

download_file(
    "/tmp/full_load_sample_5_cities_2024.json.gz",
    "full_load_sample_5_cities_2024.json.gz"
)

download_file(
    "/tmp/incremental_load_sample_5_cities_2025_01_01_to_07.json.gz",
    "incremental_load_sample_5_cities_2025_01_01_to_07.json.gz"
)